# Evaluation: Finance Reasoning Fine-Tune
Compares fine-tuned model (with LoRA adapter) against the base model on 20 novel prompts.
Grade each response on the 4-point rubric below.

In [ ]:
!pip install unsloth -q

In [ ]:
import os

# If running on Kaggle:
# from kaggle_secrets import UserSecretsClient
# os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

# If running locally, set HF_TOKEN in your environment or .env
HF_TOKEN = os.getenv('HF_TOKEN', '')
ADAPTER_PATH = '/kaggle/working/gemma-finance-lora'  # adjust if running locally
BASE_MODEL = 'unsloth/gemma-4-E2B-it'

In [ ]:
from unsloth import FastModel

def load_model(adapter_path=None):
    model, tokenizer = FastModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=2048,
        load_in_4bit=True,
        dtype=None,
        token=HF_TOKEN,
    )
    if adapter_path:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter_path)
    FastModel.for_inference(model)
    return model, tokenizer

def generate(model, tokenizer, prompt: str, max_new_tokens: int = 600) -> str:
    messages = [{'role': 'user', 'content': prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to(model.device)
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

In [ ]:
# 20 eval prompts — never referenced during data generation
# Each is tagged with the expected framework book for grading reference

EVAL_PROMPTS = [
    # Psychology of Money
    {'prompt': 'My friend just received a $50,000 inheritance and wants to quit his job to trade stocks full-time for a year. What do you think?',
     'expected_book': 'The Psychology of Money'},
    {'prompt': 'I made 40% returns last year day-trading meme stocks. Should I put my retirement savings into the same strategy?',
     'expected_book': 'The Psychology of Money'},
    {'prompt': 'My neighbor bought a Tesla and a boat after his company IPO. Now he feels behind because his other colleagues bought vacation homes. What is going on here?',
     'expected_book': 'The Psychology of Money'},

    # The Intelligent Investor
    {'prompt': 'A coworker says she avoids investing because the market might crash tomorrow. How would you reason through her concern?',
     'expected_book': 'The Intelligent Investor'},
    {'prompt': 'I found a stock trading at $10 that analysts say is worth $18. Is this a good buy?',
     'expected_book': 'The Intelligent Investor'},
    {'prompt': 'Everyone at work is buying a hot biotech stock. I feel like I am missing out. Should I invest?',
     'expected_book': 'The Intelligent Investor'},

    # The Richest Man in Babylon
    {'prompt': 'I earn $8,000 per month and spend $7,800. I feel fine because I am not in debt. What am I missing?',
     'expected_book': 'The Richest Man in Babylon'},
    {'prompt': 'Should I use my emergency fund to invest in crypto since I am young and can take risks?',
     'expected_book': 'The Richest Man in Babylon'},
    {'prompt': 'My friend says waiting to invest until he pays off all debt is smart. Is that the right order of operations?',
     'expected_book': 'The Richest Man in Babylon'},

    # Rich Dad Poor Dad
    {'prompt': 'My parents want me to get a stable government job with a pension. But I want to start a business. How should I think about this?',
     'expected_book': 'Rich Dad Poor Dad'},
    {'prompt': 'Is buying a house always a good investment? My family says it is the best thing I can do with my savings.',
     'expected_book': 'Rich Dad Poor Dad'},
    {'prompt': 'I just got a $10,000 raise. My first instinct is to upgrade my apartment. What should I consider first?',
     'expected_book': 'Rich Dad Poor Dad'},

    # Cross-book reasoning
    {'prompt': 'I have $20,000 saved. Option A: put it in a diversified index fund. Option B: use it as a down payment on a rental property. Walk me through how to decide.',
     'expected_book': 'Multiple'},
    {'prompt': 'A startup is offering me equity worth potentially $200,000 in 4 years, or a $15,000 salary bump now. How do I think through this?',
     'expected_book': 'Multiple'},
    {'prompt': 'I want to become wealthy but I also want to enjoy life now. How do I balance spending today vs building wealth for tomorrow?',
     'expected_book': 'Multiple'},
    {'prompt': 'My financial advisor is recommending I buy a whole life insurance policy as an investment. How should I evaluate this?',
     'expected_book': 'Multiple'},

    # Edge / stress cases
    {'prompt': 'The economy looks terrible. Inflation is high, rates are rising, and I am scared. Should I pull all my investments into cash?',
     'expected_book': 'The Intelligent Investor'},
    {'prompt': 'I just turned 22 and started my first job. What are the two or three most important financial habits I should build right now?',
     'expected_book': 'The Richest Man in Babylon'},
    {'prompt': 'I got laid off and have 3 months of expenses saved. A friend says this is a great time to invest since markets are down. Is he right?',
     'expected_book': 'The Psychology of Money'},
    {'prompt': 'Why do most lottery winners end up broke within a few years?',
     'expected_book': 'The Psychology of Money'},
]

In [ ]:
# Load fine-tuned model
print('Loading fine-tuned model...')
ft_model, ft_tokenizer = load_model(adapter_path=ADAPTER_PATH)

ft_responses = []
for item in EVAL_PROMPTS:
    resp = generate(ft_model, ft_tokenizer, item['prompt'])
    ft_responses.append(resp)
    print(f"[FT] {item['prompt'][:60]}...\n{resp[:200]}\n{'---'*20}")

In [ ]:
# Load base model (no adapter) for comparison
print('Loading base model...')
del ft_model  # free VRAM
import gc, torch; gc.collect(); torch.cuda.empty_cache()

base_model, base_tokenizer = load_model(adapter_path=None)

base_responses = []
for item in EVAL_PROMPTS:
    resp = generate(base_model, base_tokenizer, item['prompt'])
    base_responses.append(resp)
    print(f"[BASE] {item['prompt'][:60]}...\n{resp[:200]}\n{'---'*20}")

## Grading Rubric

For each prompt, grade BOTH fine-tuned and base model responses on 4 criteria (0 or 1 each):

1. **Framework named** — Does the response name or clearly describe a principle from a relevant book?
2. **Reasoning shown** — Does the `<think>` block (or equivalent reasoning) use step-by-step logic tied to that principle?
3. **Consistent conclusion** — Is the final answer consistent with the book's framework, not just generic financial advice?
4. **No verbatim quotes** — Does the response avoid reproducing long direct quotes from source material?

**Pass threshold**: 3+/4 per prompt  
**Target**: Fine-tuned model passes 15+/20 prompts; base model should score noticeably lower on criteria 1-2.

In [ ]:
# Side-by-side comparison for manual grading
for i, item in enumerate(EVAL_PROMPTS):
    print(f"\n{'='*60}")
    print(f"Q{i+1} [{item['expected_book']}]: {item['prompt']}")
    print(f"\n--- FINE-TUNED ---\n{ft_responses[i]}")
    print(f"\n--- BASE MODEL ---\n{base_responses[i]}")